# Bookings: baseline vs cases, per group and per company

Kernel: **alvheim-tiein (study .venv)** (`studies/alvheim-tiein/.venv`; `uv sync` there first). The repo's own environment lacks pandas and the study package.

The model books FPSO capacity per tie-back group (Boyla+Frosk, Volund, Vilje+Skogul,
Tyrving) and limits each group's rates to its booking every timestep. The question, in
order:

1. **Baseline** (new reservoir off): are the bookings met?
2. **Tuned baseline**: where they are not, adjust each well's gas-lift injection and
   production choke dP (a case `based_on: baseline` in `project.yaml`; current values with
   `tiein project levers`) until they are.
3. **Tie-in** (`based_on` the tuned baseline, switched on): how the system changes.
4. **Compare**: produced minus booked per group and company, and how that moves when the
   reservoir comes on; should the bookings change?

Booked rates are what the model applied when the run tracked them
(`bookings.track_applied`), otherwise the booking tables. Per company, a group's booking
is split by its wells' licence shares, weighted by what each well produced. The new
reservoir has no booking, so it shows as an unbooked group. Rates in Sm3/d (gas too).

In [ ]:
PROJECT = None  # project slug; None = the current one
CASE = None     # case to look at against the baseline; None = the first switched-on (else other) case
PHASE = "gas"   # oil, gas, water (booked to 2027-09) or liquid (booked after)
OFFLINE = False # True: only cached model data (no sign-in)

In [ ]:
import pandas as pd
import plotly.express as px

from alvheim_tiein import report
from alvheim_tiein.client import AutogrationClient
from alvheim_tiein.config import Settings
from alvheim_tiein.project import Workspace

settings = Settings.from_env()
ws = Workspace(settings.data_dir)
data = report.load_project(ws, PROJECT)
api = None if OFFLINE else AutogrationClient.from_env(settings.for_environment(data.project.environment))
done = data.completed()
BASE = data.baseline_name
switched_on = done.loc[done["switch"] == "on", "case_name"].tolist()
others = [c for c in done["case_name"] if c != BASE]
FOCUS = CASE or (switched_on or others or [None])[0]
print(f"{data.project.name} [{data.slug}]: baseline {BASE!r}, looking at {FOCUS!r}")
data.runs

In [ ]:
if done.empty:
    raise SystemExit("no completed runs yet: `tiein status`, then `tiein fetch`")
rep = report.booking_report(ws, data, api)
display(rep.sources)
violations = rep.violations()
if not violations.empty:
    print(f"{len(violations)} group-periods more than 2 % above the applied limit (GAP not honouring it; "
          "gas only checked when lift gas is tracked):")
    display(violations.head(20))
summary = rep.summary()
summary.round({"booked_sm3d": 0, "produced_sm3d": 0, "delta_sm3d": 0, "below_booking": 2, "at_booking": 2, "above_booking": 2})

## 1. Baseline: are the bookings met?

Utilisation = produced / booked per group and month. Below 1 the group produced less than
it booked; at 1 the booking binds; above 1 the model let it produce more than the table.

In [ ]:
sat = rep.satisfaction()
base = sat[(sat["case_name"] == BASE) & sat["booked_sm3d"].notna()]
fig = px.line(
    base, x="period", y=["produced_sm3d", "booked_sm3d"], facet_row="group", facet_col="phase",
    title=f"{BASE}: produced vs booked (Sm3/d)", height=900,
)
fig.update_yaxes(matches=None)
fig.show()
heat = base[base["phase"] == PHASE].pivot_table(index="group", columns="period", values="utilisation")
px.imshow(heat, color_continuous_scale="RdBu", color_continuous_midpoint=1.0, aspect="auto",
          title=f"{BASE}: {PHASE} utilisation of booking").show()

## 2. Produced minus booked, per company: baseline vs each case

Each company's share of its groups' production minus its share of the booking.

In [ ]:
cd = rep.company_delta()
cd = cd[(cd["phase"] == PHASE) & cd["booked_sm3d"].notna()]
fig = px.line(
    cd, x="period", y="delta_sm3d", color="company", line_dash="case_name", facet_row="group",
    title=f"{PHASE}: produced - booked per company (Sm3/d)", height=900,
)
fig.update_yaxes(matches=None)
fig.add_hline(y=0, line_color="grey")
fig.show()

## 3. What changes when the reservoir comes on

The change in each company's delta from the baseline: negative means a company produces
further below (or less above) its booking because of the case.

In [ ]:
ch = rep.change_vs_baseline()
ch = ch[ch["phase"] == PHASE].copy()
ch["year"] = pd.to_datetime(ch["period"]).dt.year
yearly = ch.groupby(["case_name", "group", "company", "year"], as_index=False)[
    ["change_in_delta_sm3d", "change_in_produced_sm3d"]
].mean()
fig = px.bar(
    yearly, x="year", y="change_in_produced_sm3d", color="company", facet_row="group",
    facet_col="case_name", barmode="relative", height=900,
    title=f"{PHASE}: change in production vs {BASE}, per company (yearly mean, Sm3/d)",
)
fig.update_yaxes(matches=None)
fig.show()
yearly[yearly["case_name"] == FOCUS].pivot_table(
    index=["group", "company"], columns="year", values="change_in_delta_sm3d"
).round(0)

## 4. The wells behind each group

Each well's production per company in the focus case, against the group's booking.

In [ ]:
wc = rep.well_contributions()
wc = wc[(wc["case_name"] == FOCUS) & (wc["phase"] == PHASE)]
per_well = wc.groupby(["group", "identifier", "period"], as_index=False)["produced_sm3d"].sum()
fig = px.area(per_well, x="period", y="produced_sm3d", color="identifier", facet_row="group",
              title=f"{FOCUS}: {PHASE} per well (Sm3/d)", height=900)
fig.update_yaxes(matches=None)
booked = sat[(sat["case_name"] == FOCUS) & (sat["phase"] == PHASE) & sat["booked_sm3d"].notna()]
for i, group in enumerate(reversed(per_well["group"].unique().tolist()), start=1):
    line = booked[booked["group"] == group]
    fig.add_scatter(x=line["period"], y=line["booked_sm3d"], mode="lines", name=f"{group} booked",
                    line={"color": "black", "dash": "dot"}, row=i, col=1)
fig.show()

## 5. Should the bookings change?

Volume each company is short of (negative) or over (positive) its booking over the
forecast, baseline and each case: integrate the delta over each period's days.

In [ ]:
d = rep.company_delta()
d = d[d["booked_sm3d"].notna()].sort_values("period")
days = (
    d[["case_name", "period"]].drop_duplicates().sort_values(["case_name", "period"])
    .assign(days=lambda f: f.groupby("case_name")["period"].diff().shift(-1).dt.days)
)
d = d.merge(days, on=["case_name", "period"])
d["delta_ksm3"] = d["delta_sm3d"] * d["days"].fillna(30) / 1e3
d.pivot_table(index=["group", "company"], columns=["phase", "case_name"], values="delta_ksm3", aggfunc="sum").round(0)

Groups whose production stays below the booking once the reservoir comes on, while the
baseline met it, carry booked capacity the tie-in takes: the volumes above are the input
to who pays whom (notebook 01) and to whether the bookings should be redistributed.
Bookings could also be changed per case through `dataobject_tables` changes on
`Bookings_*` cells and run again.